# Model Evaluation — Content Completion Prediction

Confusion matrix, precision/recall, feature importance.

**Reads:** `gold_ml_features` + saved model  **Writes:** `gold_ml_feature_importance`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when, isnan
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassificationModel

spark = SparkSession.builder.getOrCreate()
model = RandomForestClassificationModel.load('Files/models/completion_rf')
print('Model loaded')

In [ ]:
df = spark.read.table('gold_ml_features')
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

numeric_features = [
    'duration_mins', 'release_year', 'monthly_fee',
    'view_hour', 'view_dow', 'is_weekend',
]
cat_cols = ['genre', 'content_type', 'production_cost_bucket', 'language',
            'plan_type', 'region', 'age_group', 'device_type']
indexed_df = df
cat_idx_cols = []
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)
    cat_idx_cols.append(idx_col)

all_features = numeric_features + cat_idx_cols
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(indexed_df).select('features', col('had_complete').cast('double').alias('label'))
_, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
predictions = model.transform(test_df)
print(f'Test predictions: {predictions.count():,} rows')

In [ ]:
print('=== Confusion Matrix ===')
predictions.groupBy('label', 'prediction').count().orderBy('label', 'prediction').show()
tp = predictions.filter((col('label') == 1) & (col('prediction') == 1)).count()
fp = predictions.filter((col('label') == 0) & (col('prediction') == 1)).count()
fn = predictions.filter((col('label') == 1) & (col('prediction') == 0)).count()
tn = predictions.filter((col('label') == 0) & (col('prediction') == 0)).count()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
print(f'TP={tp} FP={fp} FN={fn} TN={tn}')
print(f'Precision: {precision:.4f}  Recall: {recall:.4f}')

In [ ]:
importances = model.featureImportances.toArray()
rows = sorted(
    zip(all_features, [float(importances[i]) if i < len(importances) else 0.0
                       for i in range(len(all_features))]),
    key=lambda r: r[1], reverse=True,
)
print('=== Top 10 Features ===')
for name, imp in rows[:10]:
    print(f'  {name:30s} {imp:.4f}')
fi_spark = spark.createDataFrame(rows, ['feature', 'importance']).withColumn('model_timestamp', current_timestamp())
fi_spark.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_feature_importance')
print('Feature importance saved')

In [ ]:
spark.sql('OPTIMIZE gold_ml_feature_importance')
print('Evaluation complete')